In [ ]:
from src.generation import (
    split_points,
    vector_to_coordinate,
    apply_dataseries_to_polygon,
)
from src.dataseries import DataSeries
from src.spheroid import Spheroid
from src.projection import UVMap
from src.tectonic_sim import TectonicSimulation
import noise

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection
import pyvista as pv
import panel as pn

pn.extension("vtk")

In [ ]:
grid_size = [7] * 3
centering_translation = np.array([-size / 2 for size in grid_size], dtype=float)
sampling_translation = np.array([4.0, -5.0, 0.0], dtype=float)
arr = (
    DataSeries.three_d_perlin(
        *grid_size,
        seed=42,
        resolution=0.20,
        scale=4.0,
        translation=sampling_translation,
        repeat=(255, 255, 255),
    )
    .translate_data(centering_translation - sampling_translation)
    .cull_below_threshold(threshold=-0.25)
    .cull_within_radius(radius=1, center_point=(0.0, 0.0, 0.0))
)

# only now split for plotting
coords, values = split_points(arr.data)

x, y, z = coords.T
for rot_deg in range(0, 360, 45):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    sc = ax.scatter(x, y, z, c=values, cmap="plasma", s=20)
    ax.view_init(elev=30, azim=rot_deg)
    plt.colorbar(sc, ax=ax, label="Perlin value")
    plt.show()

In [ ]:
# unit vectors from arr with coordinate_to_vector
# coordinate_to_vector only accepts one coordinate at a time,
# do no use raw python loops
# arr shape is [[z,y,z],value]
# from src.utils import coordinate_to_vector


# unit_vectors = np.array([coordinate_to_vector(coord) for coord in arr.data[:, 0]])
# coords, values = split_points(unit_vectors)
# x, y, z = coords.T
# for rot_deg in range(0, 360, 45):
#      fig = plt.figure()
#      ax = fig.add_subplot(projection="3d")
#      sc = ax.scatter(x, y, z, c=values, cmap="plasma", s=20)
#      ax.view_init(elev=30, azim=rot_deg)
#      plt.colorbar(sc, ax=ax, label="Perlin value")
#      plt.show()

In [ ]:
# ico = icosahedron(radius=1.0)
# vectors = np.stack(ico[0])

#

# vec_to_coord = np.vectorize(lambda d, m: vector_to_coordinate([d, m]), otypes=[np.ndarray])
# coords = np.vstack(vec_to_coord(vectors[:, 0], vectors[:, 1]))

# values = vectors[:, 1]
# print(coords.shape, values.shape)

# for rot_deg in range(0, 360, 45):
#      fig = plt.figure()
#      ax = fig.add_subplot(projection="3d")
#      sc = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=values, cmap="plasma", s=20)
#      ax.view_init(elev=30, azim=rot_deg)
#      plt.colorbar(sc, ax=ax, label="Perlin value")
#      plt.show()

In [ ]:
ico = Spheroid(radius=5.0, subdivisions=4)
vectors, edges, faces = ico.vectors, ico.edges, ico.faces
# convert to ndarray form3)
vectors = np.stack(vectors)

# --- normalize all vectors in bulk ---
directions = np.stack(vectors[:, 0]).astype(float)
magnitudes = vectors[:, 1].astype(float)

norms = np.linalg.norm(directions, axis=1, keepdims=True)
safe_norms = np.where(norms == 0, 1.0, norms)
normalized_dirs = directions / safe_norms
normalized_mags = np.ones_like(magnitudes, dtype=float)

vectors = np.empty((len(normalized_dirs), 2), dtype=object)
vectors[:, 0] = [d.astype(np.float32) for d in normalized_dirs]
vectors[:, 1] = normalized_mags

# --- convert to coordinates in bulk ---
coords = normalized_dirs * normalized_mags[:, None]
values = normalized_mags

# --- plotting ---
for rot_deg in range(0, 360, 90):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    # scatter vertices
    sc = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        coords[:, 2],
        c=values,
        cmap="plasma",
        s=50,
        depthshade=True,
    )

    # edges
    edge_lines = np.stack([coords[edges[:, 0]], coords[edges[:, 1]]], axis=1)
    ax.add_collection3d(Line3DCollection(edge_lines, colors="black", linewidths=1))

    # faces
    face_polys = [coords[face] for face in faces]
    ax.add_collection3d(
        Poly3DCollection(
            face_polys, facecolors="cyan", edgecolors="k", linewidths=0.5, alpha=0.2
        )
    )

    ax.view_init(elev=30, azim=rot_deg)
    ax.set_box_aspect([1, 1, 1])

    plt.colorbar(sc, ax=ax, label="Magnitude")
    plt.show()

In [ ]:
resolution = (256, 256)
uv_map = UVMap.from_spheroid(resolution, ico, uv_padding=0.125)

In [ ]:
# plot the unwrapped faces and their connections
projection = uv_map.projection
faces_2d, vertices_2d, vertex_map = (
    projection.faces,
    projection.vertices,
    projection.vertex_map,
)
plt.figure(figsize=(4, 4))
for face in faces_2d:
    poly = vertices_2d[np.array(face, dtype=int)]
    plt.fill(poly[:, 0], poly[:, 1], edgecolor="black", fill=False, linewidth=0.5)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Unwrapped faces of spheroid")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()

In [ ]:
creation_configs = {
    "density_range": (0.80, 0.95),
    "velocity_range": (-1.6, 1.6),
    "height_range": (-0.03, 0.06),
    "velocity_decay": 0.04,
    "density_decay": 0.03,
}

setup_configs = {
    "velocity_amplification": 2.4,
    "friction_dampening": 1.1,
    "contributing_neighbor_radius": 2,
    "base_friction": 1.0,
    "position_weight": 0.55,
    "velocity_weight": 0.06,
    "density_weight": 1.2,
}

run_configs = {
    "stress_thresholds": np.arange(0.9, 6.3, 0.9),
    "stress_propagation_radius_per_threshold": 1,
    "stress_distribution_factor": 0.24,
}

region_count = 9

# --- simulation ---
techtonic_sim = TectonicSimulation.from_spheroid(
    ico, region_count, seed=69, **creation_configs
)
techtonic_sim.setup_simulation(**setup_configs)
techtonic_sim.run(cycles=10, **run_configs)
planetoid = techtonic_sim.apply_to_spheroid(ico.clone(), stress_scalar=0.25)

coords = np.asarray(planetoid.vertices, dtype=float)
edges_array = np.asarray(
    planetoid.edges, dtype=int
)  # kept if you want edge overlays later
faces_index_arrays = [
    np.asarray(face_indices, dtype=int) for face_indices in planetoid.faces
]

In [ ]:
# --- per-face scalars (nodes correspond 1:1 to faces) ---
face_stress_values = np.array(
    [node.stress for node in techtonic_sim.nodes], dtype=float
)
face_region_values = np.array([node.region for node in techtonic_sim.nodes], dtype=int)
min_stress_value = float(face_stress_values.min())
max_stress_value = float(face_stress_values.max())

# --- pack polygon faces for PyVista (no triangulation) ---
faces_packed_parts = []
for face_indices in faces_index_arrays:
    vertex_count_in_face = np.array([len(face_indices)], dtype=np.int64)
    faces_packed_parts.append(vertex_count_in_face)
    faces_packed_parts.append(face_indices.astype(np.int64))
faces_packed = np.concatenate(faces_packed_parts)

# --- build polydata ---
poly_mesh = pv.PolyData(coords, faces=faces_packed)
poly_mesh.cell_data.clear()
poly_mesh.cell_data["stress"] = face_stress_values
poly_mesh.cell_data["region"] = face_region_values

# --- choose notebook backend (interactive) ---
try:
    pv.set_jupyter_backend("panel")
except Exception:
    pv.set_jupyter_backend("static")

# --- plotter + interactive controls (Panel) ---
plotter = pv.Plotter()


def render_view(scalar_name: str, colormap_name: str, show_edges_flag: bool) -> None:
    """Rebuild the scene with the chosen scalar and style."""
    camera_position = plotter.camera_position
    plotter.clear()
    scalar_title = "Stress" if scalar_name == "stress" else "Region"
    color_limits = (
        (min_stress_value, max_stress_value)
        if scalar_name == "stress"
        else (0, region_count - 1)
    )

    plotter.add_mesh(
        poly_mesh,
        scalars=scalar_name,
        cmap=colormap_name,
        clim=color_limits,
        show_edges=show_edges_flag,
        edge_color="black" if show_edges_flag else None,
        smooth_shading=False,
        scalar_bar_args={"title": scalar_title},
    )
    plotter.add_axes()
    if camera_position is not None:
        plotter.camera_position = camera_position
    plotter.render()


# initial draw
render_view(scalar_name="stress", colormap_name="plasma", show_edges_flag=True)
viewer = plotter.show(return_viewer=True)

# widgets
scalar_selector = pn.widgets.RadioButtonGroup(
    name="Scalar", options=["stress", "region"], value="stress"
)
colormap_selector = pn.widgets.Select(
    name="Colormap",
    options=["plasma", "viridis", "inferno", "magma", "cividis", "tab20"],
    value="plasma",
)
edges_toggle = pn.widgets.Checkbox(name="Show edges", value=True)


def on_controls_change(event=None):
    render_view(
        scalar_name=scalar_selector.value,
        colormap_name=colormap_selector.value,
        show_edges_flag=edges_toggle.value,
    )


for widget in (scalar_selector, colormap_selector, edges_toggle):
    widget.param.watch(lambda _evt: on_controls_change(), "value")

layout = pn.Row(
    pn.Column(
        "### Controls",
        scalar_selector,
        colormap_selector,
        edges_toggle,
        sizing_mode="fixed",
        width=250,
    ),
    viewer,
)
layout

In [ ]:
projection_configs = {
    # "weights_exponent": 1.0,
    # "plane_relaxation": 1.0,
    "scaler": 1.0 - 0.02
}


def noise_wrapper(x, y, z):
    return noise.pnoise3(x, y, z, repeatx=1024, repeaty=1024, repeatz=1024, base=42)


uv_map.generate_noise(noise_function=noise_wrapper, spheroid=ico, **projection_configs)

In [ ]:
# rasterize: this gives pixel coordinates + face indices
pixel_coords, face_indices = uv_map.coords, uv_map.face_indices
pixel_values = uv_map.values


# Unpack for plotting
us = pixel_coords[:, 0]
vs = pixel_coords[:, 1]
vals = pixel_values


plt.figure(figsize=(6, 6))
plt.scatter(us, vs, c=vals, s=1, vmin=-1, vmax=1, cmap="viridis")
plt.gca().set_aspect("equal", adjustable="box")
plt.colorbar(label="Value")
plt.xlabel("U")
plt.ylabel("V")
plt.title("UV raster of spheroid")
plt.show()

In [ ]:
from src.utils import coordinate_to_vector


projection = uv_map.projection

# map all UV pixels to 3D
coords_3d = np.asarray(
    [
        projection.project_to_spheroid(
            int(u), int(v), int(fi), planetoid, **projection_configs
        )
        for (u, v), fi in zip(uv_map.coords, uv_map.face_indices)
    ],
    dtype=float,
)

# value that corresponds to each pixel
values = uv_map.values
# plot the initial ico along side the projected points

vectors, edges, faces = planetoid.vectors, planetoid.edges, planetoid.faces
noise_scalar = 0.8
# coords = np.asarray([vector_to_coordinate(vec) for vec in vectors], dtype=float)
# sum the value with the vector's magnitude before converting to coordinates
coords = []
fig_values = []
for i, vertex in enumerate(coords_3d):
    new_vertex = vertex.copy()
    new_vector = coordinate_to_vector(new_vertex)
    new_vector[1] += values[i] * noise_scalar
    fig_values.append(new_vector[1])
    coords.append(vector_to_coordinate(new_vector))
coords = np.asarray(coords)
fig_values = np.asarray(fig_values)

edges = np.asarray(edges, dtype=int)
faces = [np.asarray(face, dtype=int) for face in faces]


for rot_deg in range(0, 360, 45):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    # edge_lines = np.stack([coords[edges[:, 0]], coords[edges[:, 1]]], axis=1)
    # ax.add_collection3d(Line3DCollection(edge_lines, colors="black", linewidths=1))
    sc = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        coords[:, 2],
        c=fig_values,
        cmap="viridis",
        s=1,
        depthshade=True,
    )

    # edges

    # faces (note: fills assume planarity)
    # face_polys = [coords[face] for face in faces]
    # ax.add_collection3d(Poly3DCollection(face_polys,
    #                                      facecolors="cyan", edgecolors="k",
    #                                      linewidths=0.5, alpha=1.0))

    ax.view_init(elev=30, azim=rot_deg)
    ax.set_box_aspect([1, 1, 1])

    plt.colorbar(sc, ax=ax, label="Magnitude")
    plt.show()

In [ ]:
default_value = -10.0
vectors, edges, faces = apply_dataseries_to_polygon(
    dataseries=arr.data,
    polygon=(ico.vectors, ico.edges, ico.faces),
    default=default_value,
    # ray_radius=0.5,
    # ray_step=0.25,
    # max_march=2.5
)
print("completed applying data series on polygon...")
r = 1.0
scaled_vectors = np.array(
    [np.array([v[0], r, v[2]], dtype=object) for v in vectors],
    dtype=object,
)
print("completed scaling vectors...")

# convert to coordinates using helper
coords = np.stack(
    [vector_to_coordinate(v[:2], origin_point=(0, 0, 0)) for v in scaled_vectors]
)
values = np.array([v[2] for v in scaled_vectors], dtype=float)
print("completed converting to coordinates...")

# build face geometry and average colors
face_coords = [coords[f] for f in faces]
face_colors = np.array([values[f].mean() for f in faces])
min_legend_value = min(-0.5, face_colors.min())
print("completed building face geometry and colors...")

print("Start plotting...")
for rot_deg in range(0, 360, 90):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    # edges
    edge_lines = np.stack([coords[edges[:, 0]], coords[edges[:, 1]]], axis=1)
    ax.add_collection3d(Line3DCollection(edge_lines, colors="black", linewidths=1))
    norm = mpl.colors.Normalize(vmin=min_legend_value, vmax=face_colors.max())
    facecolors = plt.cm.plasma(norm(face_colors))
    ax.add_collection3d(
        Poly3DCollection(
            face_coords,
            facecolors=facecolors,
            edgecolors="k",
            linewidths=0.5,
            alpha=1.0,
        )
    )

    ax.view_init(elev=30, azim=rot_deg)
    ax.set_box_aspect([1, 1, 1])

    mappable = mpl.cm.ScalarMappable(cmap="plasma")
    mappable.set_array(face_colors)
    # set the min value on the legend to -5 or face_colors whichever is lower
    # use the min_legend_value as the minimum for the colorbar
    plt.colorbar(mappable, ax=ax, label="Face mean value")

    plt.show()